# WLS-Var investigation — sims + real-data biorep-split

Builds on `tests/benchmark/bench_linear_weights.py` with a **peptide-level generative model**
(protein × peptide × timepoint) and a **real-data biorep-split** validation. Estimator defs and the
eBayes moderation: `reports/2026-07-24_linear_wls_per_point_var.md`.

`simulate_protein`: each protein has `n_pep` peptides with an intrinsic k-offset **shared across
conditions** (paired design → cancels in Δk). *collapse* = one inverse-variance θ per (biorep, t)
cell (`Var=1/Σ(1/σ²)`, dispersion-biased); *pooled* = every peptide-point (unbiased var, but
pseudoreplicated → **never a default**). Metric: `reject` = Δk rejection rate (Type-I when kA=kB).

In [1]:
import sys; sys.path.insert(0, "..")
import numpy as np, pandas as pd
from wls_var_sim import run, d0_grid, fit_fdist, simulate_protein, collapse, biorep_split
pd.set_option("display.width", 140); NSIM = 150

## Scenario 2 — collapse vs pooled (`n_pep=4`)

In [2]:
print("TYPE-I  (kA=kB=0.10)"); display(run(0.10, 0.10, 4, NSIM, 1))
print("\nPOWER   (kA=0.10, kB=0.14)"); display(run(0.10, 0.14, 4, NSIM, 3))

TYPE-I  (kA=kB=0.10)


,method,rollup,n,dk_bias,dk_rmse,reject
0,wls_fit,collapse,150,0.000451,0.002979,0.046667
1,wls_fit,pooled,150,-0.000016,0.003240,0.000000
2,wls_fit_var,collapse,150,0.000387,0.003879,0.086667
3,wls_fit_var,pooled,150,0.000416,0.004297,0.060000
4,ebayes,collapse,150,0.000475,0.003408,0.080000
5,ebayes,pooled,150,0.000384,0.002879,0.006667



POWER   (kA=0.10, kB=0.14)


,method,rollup,n,dk_bias,dk_rmse,reject
0,wls_fit,collapse,150,0.000829,0.008295,1.0
1,wls_fit,pooled,150,0.003365,0.008442,1.0
2,wls_fit_var,collapse,150,0.000953,0.009164,1.0
3,wls_fit_var,pooled,150,0.002339,0.009677,1.0
4,ebayes,collapse,150,0.000904,0.008872,1.0
5,ebayes,pooled,150,0.002500,0.007762,1.0


`wls_fit` ~nominal (collapse) / conservative (pooled); raw `wls_fit_var` inflates Type-I under both;
**eBayes fixes it, pooled+eBayes safest** — but pooled pseudoreplicates peptides, so it stays an
*investigation* arm, never a default (rigorous stats want biological replicates, not repeated peptides).

## Scenario 1 — single-peptide proteins (`n_pep=1`)

In [3]:
display(run(0.10, 0.10, 1, NSIM, 2))

,method,rollup,n,dk_bias,dk_rmse,reject
0,wls_fit,collapse,150,0.000018,0.006092,0.020000
1,wls_fit,pooled,150,0.000018,0.006092,0.020000
2,wls_fit_var,collapse,150,0.000415,0.008098,0.126667
3,wls_fit_var,pooled,150,0.000415,0.008098,0.126667
4,ebayes,collapse,150,0.000242,0.006765,0.053333
5,ebayes,pooled,150,0.000242,0.006765,0.053333


Per-point weighting **hurts** (RMSE↑, Type-I↑, no gain); **eBayes shrinks back to `wls_fit`** and
protects the few-peptide cases. The moderated estimator correctly does ≈nothing here.

## d₀ frontier + Smyth `fitFDist` (with limma-robust Winsorization)

Is the data-estimated d₀ optimal, or just sensible? And do heavy-tailed variances need robustifying?

In [4]:
print("eBayes d0 sweep (collapse, TYPE-I at kA=kB=0.10)"); display(d0_grid(0.10, 0.10, 4, NSIM, 1))
# robustness of fitFDist to outlier variances (limma robust=TRUE flavour), boomi collapsed cells:
from wls_var_sim import collapse_cells
c = collapse_cells(pd.read_table("../runs/boomi_ipsc_d2o/riana_fit_fractions.txt", comment="#"))
for trim in (0.0, 0.05, 0.10):
    d0, s0 = fit_fdist(c["var"].to_numpy(), c["df"].to_numpy(), trim=trim)
    print(f"  fitFDist trim {trim:.0%}: d0 = {d0:.2f}")

eBayes d0 sweep (collapse, TYPE-I at kA=kB=0.10)


,d0,reject,dk_rmse
0,0.5,0.093333,0.003660
1,1.0,0.086667,0.003543
2,2.0,0.080000,0.003408
3,4.0,0.066667,0.003278
4,8.0,0.060000,0.003168


  fitFDist trim 0%: d0 = 1.44
  fitFDist trim 5%: d0 = 1.58
  fitFDist trim 10%: d0 = 1.81


`fitFDist` returns a sensible d₀ but the raw estimate under-shrinks on heavy-tailed proteomics
variances — **Winsorizing (limma `robust=TRUE`) raises d₀ ~25%** (1.44→1.81 on boomi), the safer value.
Production should use the robust estimate (or fall back to fixed d₀=2).

## Real-data validation — biorep-split stability

No ground truth, so use the **biological replicates**: fit per-protein k separately per biorep, and
measure `|log(k_b1/k_b2)|` within each (protein, condition) under `wls` vs `wls-var` (eBayes, robust
d₀). **Lower = more consistent across biological replicates = better** — a fairer test than the
equal-weight per-peptide median (which is itself just an estimator, not truth; that lve test was
inconclusive).

In [5]:
for run_dir in ["boomi_ipsc_d2o", "boomi_ipsc_o18", "timeseries_lauren5_7_ipsc_mesoderm_o18", "timeseries_dia"]:
    tbl, d0 = biorep_split(f"../runs/{run_dir}", d0_trim=0.10)
    if tbl is None: print(f"{run_dir}: <2 bioreps"); continue
    imp = (tbl.loc['wls','multi_pep'] - tbl.loc['wls-var','multi_pep']) / tbl.loc['wls','multi_pep'] * 100
    print(f"### {run_dir}  (robust d0={d0:.2f})   multi-peptide biorep consistency: {imp:+.1f}%")
    display(tbl.round(4))

### boomi_ipsc_d2o  (robust d0=1.81)   multi-peptide biorep consistency: +10.7%


,all,multi_pep,single_pep,n
wls,0.1001,0.0768,0.1866,2747.0
wls-var,0.0926,0.0686,0.1881,2747.0


### boomi_ipsc_o18  (robust d0=1.78)   multi-peptide biorep consistency: +10.4%


,all,multi_pep,single_pep,n
wls,0.0890,0.0599,0.1766,2572.0
wls-var,0.0796,0.0537,0.1777,2572.0


### timeseries_lauren5_7_ipsc_mesoderm_o18  (robust d0=1.61)   multi-peptide biorep consistency: +9.9%


,all,multi_pep,single_pep,n
wls,0.1743,0.1221,0.3056,5749.0
wls-var,0.1561,0.1100,0.3133,5749.0


### timeseries_dia  (robust d0=1.59)   multi-peptide biorep consistency: +2.1%


,all,multi_pep,single_pep,n
wls,0.1196,0.1176,0.1692,1887.0
wls-var,0.1168,0.1151,0.1806,1887.0


**Finding — the benefit scales with curve richness.** `wls-var` improves multi-peptide biorep
consistency **~10%** on the rich 9–12-timepoint sets (boomi D₂O + ¹⁸O, lauren mesoderm ¹⁸O), but only
**~2%** on `timeseries_dia` (a sparse **3-timepoint, no-t0** DIA set) — with ~3 points per curve there
is little per-point heteroscedasticity to exploit, and the biorep noise is dominated by sparse-curve
estimation error (confounded with DIA acquisition). It is safe/neutral on single-peptide throughout.
`juber_ac16_d2o` is 1-biorep (can't split). Net: a real green light for `wls-var` as **opt-in** (robust
d₀, eBayes protecting single-peptide), with the honest caveat that the gain concentrates on rich,
multi-peptide time series — the linear Δk model's main use case.